In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/bank_customer_churn_prediction.csv")
df.columns, df.head(10)

(Index(['customer_id', 'credit_score', 'country', 'gender', 'age', 'tenure',
        'balance', 'products_number', 'credit_card', 'active_member',
        'estimated_salary', 'churn'],
       dtype='object'),
    customer_id  credit_score  country  gender  age  tenure    balance  \
 0     15634602           619   France  Female   42       2       0.00   
 1     15647311           608    Spain  Female   41       1   83807.86   
 2     15619304           502   France  Female   42       8  159660.80   
 3     15701354           699   France  Female   39       1       0.00   
 4     15737888           850    Spain  Female   43       2  125510.82   
 5     15574012           645    Spain    Male   44       8  113755.78   
 6     15592531           822   France    Male   50       7       0.00   
 7     15656148           376  Germany  Female   29       4  115046.74   
 8     15792365           501   France    Male   44       4  142051.07   
 9     15592389           684   France    Male   27

In [3]:
# 식별자 ID 제거
df = df.drop(columns=['customer_id'], errors='ignore')

### Label Encoding 코드

In [4]:
from sklearn.preprocessing import LabelEncoder

# 범주형 컬럼 지정
label_cols = ['gender', 'country']

le = LabelEncoder()

for col in label_cols:
    df[col] = le.fit_transform(df[col])

df[label_cols]

,gender,country
0,0,0
1,0,2
2,0,0
3,0,0
4,0,2
...,...,...
9995,1,0
9996,1,0
9997,0,0
9998,1,1


### Binning(범위 나누기, 구간화)

In [5]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 30, 40, 50, 60, 100],
    labels=['20s', '30s', '40s', '50s', '60+']
)

df['age_group'] = LabelEncoder().fit_transform(df['age_group'])

df[['age', 'age_group']]

,age,age_group
0,42,2
1,41,2
2,42,2
3,39,1
4,43,2
...,...,...
9995,39,1
9996,35,1
9997,36,1
9998,42,2


### Interaction Feature(조합 피처 추가)

- age × active_member

- age × products_number

- credit_score × active_member

- products_number × active_member

- country × active_member

- country × age_group (age binning 후)

In [6]:
from sklearn.compose import ColumnTransformer

X = df.drop(columns=['churn'])
y = df['churn']

# ============================

In [ ]:
import sys, os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent  # parent of notebooks/
sys.path.append(str(project_root))

from src.data_preprocessing import Preprocessing

processing = Preprocessing()
df = pd.read_csv("../data/bank_customer_churn_prediction.csv")

processing.fit(df)
processing.transform(df)

>>>>> final columns :  Index(['credit_score', 'country', 'gender', 'age', 'tenure', 'balance',
       'products_number', 'credit_card', 'active_member', 'estimated_salary',
       'churn', 'age_group', 'gender_Female', 'gender_Male', 'country_France',
       'country_Germany', 'country_Spain'],
      dtype='object')


,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,age_group,gender_Female,gender_Male,country_France,country_Germany,country_Spain
0,-0.326221,France,Female,0.293517,-1.041760,-0.761480,-0.911583,1,1,0.021886,1,2,1.0,0.0,1.0,0.0,0.0
1,-0.440036,Spain,Female,0.198164,-1.387538,-0.104906,-0.911583,0,1,0.216534,0,2,1.0,0.0,0.0,0.0,1.0
2,-1.536794,France,Female,0.293517,1.032908,0.489346,2.527057,1,0,0.240687,1,2,1.0,0.0,1.0,0.0,0.0
3,0.501521,France,Female,0.007457,-1.387538,-0.761480,0.807737,0,0,-0.108918,0,1,1.0,0.0,1.0,0.0,0.0
4,2.063884,Spain,Female,0.388871,-1.041760,0.221806,-0.911583,1,1,-0.365276,0,2,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1.246488,France,Male,0.007457,-0.004426,-0.761480,0.807737,1,0,-0.066419,0,1,0.0,1.0,1.0,0.0,0.0
9996,-1.391939,France,Male,-0.373958,1.724464,-0.312031,-0.911583,1,1,0.027988,0,1,0.0,1.0,1.0,0.0,0.0
9997,0.604988,France,Female,-0.278604,0.687130,-0.761480,-0.911583,0,1,-1.008643,1,1,1.0,0.0,1.0,0.0,0.0
9998,1.256835,Germany,Male,0.293517,-0.695982,-0.173319,0.807737,1,0,-0.125231,1,2,0.0,1.0,0.0,1.0,0.0


# Pipeline ============================

In [11]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent  # parent of notebooks/
sys.path.append(str(project_root))

from src.custom_preprocessing import InteractionFeaturePreProcessing

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

X = df.drop(columns=["churn"])
y = df["churn"]

numeric_cols = ["age", "tenure", "credit_score", "estimated_salary", "products_number"]
balance_col  = ["balance"] # 비 정규적인 데이터
label_cols   = ["gender", "country"]
drop_col     = ['customer_id']

transformer = ColumnTransformer([
    ('id_dropper', 'drop', [c for c in drop_col if c in X.columns]),
    ("standard_scaler", StandardScaler(), numeric_cols),
    ("robust_scaler", RobustScaler(), balance_col),
    ("oh_encoder", OneHotEncoder(sparse_output=False), label_cols)
], remainder='passthrough')

preprocessing_pipeline = Pipeline([
    ("interaction_preprocessor", InteractionFeaturePreProcessing()),
    ("transformer", transformer)
], verbose=True)

preprocessing_pipeline

,steps,"[('interaction_preprocessor', ...), ('transformer', ...)]"
,transform_input,None
,memory,None
,verbose,True
,transformers,"[('id_dropper', ...), ('standard_scaler', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
result = preprocessing_pipeline.fit_transform(X, y)

X.shape, result.shape, result